## Seminar 1: Fun with Word Embeddings (3 points)

Today we gonna play with word embeddings: train our own little embeddings, load one from gensim model zoo and use it to visualize text corpora.

This whole thing is gonna happen on top of embedding dataset.

__Requirements:__  `pip install --upgrade nltk gensim bokeh` , but only if you're running locally.

In [1]:
# download the data:
!wget https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1 -O ./quora.txt
# alternative download link: https://yadi.sk/i/BPQrUu1NaTduEw

"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [2]:
import numpy as np

with open("./quora.txt", encoding="utf-8") as file:
    data = list(file)

data[50]

"What TV shows or books help you read people's body language?\n"

__Tokenization:__ a typical first step for an NLP task is to split raw data into words.
The text we're working with is in raw format: with all the punctuation and smiles attached to some words, so a simple str.split won't do.

Let's use __`nltk`__ - a library that handles many NLP tasks like tokenization, stemming or part-of-speech tagging.

In [3]:
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

['What', 'TV', 'shows', 'or', 'books', 'help', 'you', 'read', 'people', "'", 's', 'body', 'language', '?']


In [4]:
# TASK: lowercase everything and extract tokens with tokenizer. 
# data_tok should be a list of lists of tokens for each line in data.

data_tok = [tokenizer.tokenize(i.lower()) for i in data]

print(data_tok[2])

['who', 'were', 'the', 'great', 'chinese', 'soldiers', 'and', 'leaders', 'who', 'fought', 'in', 'ww2', '?']


In [5]:
assert all(isinstance(row, (list, tuple)) for row in data_tok), "please convert each line into a list of tokens (strings)"
assert all(all(isinstance(tok, str) for tok in row) for row in data_tok), "please convert each line into a list of tokens (strings)"
is_latin = lambda tok: all('a' <= x.lower() <= 'z' for x in tok)
assert all(map(lambda l: not is_latin(l) or l.islower(), map(' '.join, data_tok))), "please make sure to lowercase the data"

In [6]:
print([' '.join(row) for row in data_tok[:2]])

["can i get back with my ex even though she is pregnant with another guy ' s baby ?", 'what are some ways to overcome a fast food addiction ?']


__Word vectors:__ as the saying goes, there's more than one way to train word embeddings. There's Word2Vec and GloVe with different objective functions. Then there's fasttext that uses character-level models to train word embeddings. 

The choice is huge, so let's start someplace small: __gensim__ is another nlp library that features many vector-based models incuding word2vec.

In [7]:
from gensim.models import Word2Vec
model = Word2Vec(data_tok, 
                 vector_size=32,      # embedding vector size
                 min_count=5,  # consider words that occured at least 5 times
                 window=5).wv  # define context as a 5-word window around the target word

# From gensim docs
# vw: This object essentially contains the mapping between words and embeddings.
# After training, it can be used directly to query those embeddings in various ways.

In [8]:
# now you can get word vectors !
model.get_vector('anything')

array([-1.6397492 , -2.1496048 , -0.38729888,  4.7309523 ,  1.3326442 ,
        1.9695989 ,  2.1148982 , -3.3453786 ,  0.16363813,  3.2127657 ,
       -0.39814544,  1.6357269 ,  4.6753902 ,  1.4079772 ,  2.7887223 ,
        0.9011622 ,  0.62082547, -0.9530093 ,  1.4768339 , -3.3209424 ,
       -3.2059162 , -0.7268359 , -1.151355  , -0.8001887 ,  1.0631994 ,
       -1.8456179 , -0.48244393, -0.83067745, -0.11639381,  0.7699914 ,
       -0.8855075 ,  0.5487406 ], dtype=float32)

In [9]:
# or query similar words directly. Go play with it!
model.most_similar('you')

[('we', 0.7813891768455505),
 ('they', 0.7472084760665894),
 ('i', 0.7273057103157043),
 ('yourself', 0.6913866400718689),
 ('one', 0.6869249939918518),
 ('someone', 0.6858266592025757),
 ('somebody', 0.6473218202590942),
 ('everybody', 0.6299600005149841),
 ('everyone', 0.6154764294624329),
 ('stranger', 0.6124937534332275)]

### Using pre-trained model

Took it a while, huh? Now imagine training life-sized (100~300D) word embeddings on gigabytes of text: wikipedia articles or twitter posts. 

Thankfully, nowadays you can get a pre-trained word embedding model in 2 lines of code (no sms required, promise).

After being downloaded for the first time (or if you manually delete it), the model is saved in the `~/gensim_data` or `%USER_PATH%/gensim_data` directory. This can be checked seting the return_path parameter to True.

In [10]:
import gensim.downloader as api
model = api.load('glove-twitter-100')

In [11]:
model.most_similar(positive=["coder", "money"], negative=["brain"])

[('broker', 0.5820155739784241),
 ('bonuses', 0.5424473285675049),
 ('banker', 0.5385112762451172),
 ('designer', 0.5197198390960693),
 ('merchandising', 0.4964233338832855),
 ('treet', 0.4922019839286804),
 ('shopper', 0.4920562207698822),
 ('part-time', 0.4912828207015991),
 ('freelance', 0.4843311905860901),
 ('aupair', 0.4796452522277832)]

### Visualizing word vectors

One way to see if our vectors are any good is to plot them. Thing is, those vectors are in 30D+ space and we humans are more used to 2-3D.

Luckily, we machine learners know about __dimensionality reduction__ methods.

Let's use that to plot 1000 most frequent words

In [12]:
words = model.index_to_key[:1000] 

print(words[::100])

['<user>', '_', 'please', 'apa', 'justin', 'text', 'hari', 'playing', 'once', 'sei']


In [13]:
# for each word, compute it's vector with model
word_vectors = np.array([model.get_vector(i) for i in words])
print(word_vectors[1])

[ 0.18205   -0.048483   0.23966    0.32099   -0.27002    0.70431
 -0.21257    0.235      0.090142   0.82141    0.37843   -0.56382
 -2.4447     0.16827    0.24685    0.28649    0.062312   0.067508
 -0.58459   -0.45414   -0.22158    0.17423   -0.35558    0.14485
  0.49089   -1.7426    -0.54306   -0.51937    0.94795   -0.41739
 -0.55238   -0.057398  -0.52663    0.62976    0.097275   0.20637
  0.46261    0.089462   0.016019  -0.53854   -1.2043     0.080287
 -0.65351    0.044617   0.79527    0.044508   0.53367    0.27444
 -0.32461   -0.053683  -0.79304    0.11       0.39762   -0.044155
  0.21701    0.27977   -0.25773    0.25085    0.39711    0.32318
  0.10245   -0.030471   0.34113    0.17971    0.44436    0.054915
  0.22461   -0.80843   -0.11052    0.42366    0.61091    0.55024
  0.21958   -0.3029     0.14545   -0.46701   -0.23945    0.035106
 -0.50933   -0.12392    0.24526    0.14758   -0.30313   -0.53052
  0.80632    0.34566   -0.24541    0.71479   -0.15985    0.40129
 -0.10851   -0.61427

In [14]:
assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 100)
assert np.isfinite(word_vectors).all()

#### Linear projection: PCA

The simplest linear dimensionality reduction method is **P**rincipial **C**omponent **A**nalysis.

In geometric terms, PCA tries to find axes along which most of the variance occurs. The "natural" axes, if you wish.

<img src="https://github.com/yandexdataschool/Practical_RL/raw/master/yet_another_week/_resource/pca_fish.png" style="width:30%">


Under the hood, it attempts to decompose object-feature matrix $X$ into two smaller matrices: $W$ and $\hat W$ minimizing _mean squared error_:

$$\|(X W) \hat{W} - X\|^2_2 \to_{W, \hat{W}} \min$$
- $X \in \mathbb{R}^{n \times m}$ - object matrix (**centered**);
- $W \in \mathbb{R}^{m \times d}$ - matrix of direct transformation;
- $\hat{W} \in \mathbb{R}^{d \times m}$ - matrix of reverse transformation;
- $n$ samples, $m$ original dimensions and $d$ target dimensions;



In [21]:
from sklearn.decomposition import PCA

# map word vectors onto 2d plane with PCA. Use good old sklearn api (fit, transform)
# after that, normalize vectors to make sure they have zero mean and unit variance
pca = PCA(n_components=2)
word_vectors_pca = pca.fit_transform(word_vectors)
print(word_vectors_pca[0])

# and maybe MORE OF YOUR CODE here :)
mean = word_vectors_pca.mean(axis=0)
std = word_vectors_pca.std(axis=0)

word_vectors_pca = (word_vectors_pca - mean) / std

[0.9333839  0.50371313]


In [20]:
assert word_vectors_pca.shape == (len(word_vectors), 2), "there must be a 2d vector for each word"
assert max(abs(word_vectors_pca.mean(0))) < 1e-5, "points must be zero-centered"
assert max(abs(1.0 - word_vectors_pca.std(0))) < 1e-2, "points must have unit variance"

#### Let's draw it!

In [22]:
import bokeh.models as bm, bokeh.plotting as pl
from bokeh.io import output_notebook
output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=600, height=400, show=True, **kwargs):
    """ draws an interactive plot for data points with auxilirary info on hover """
    if isinstance(color, str): color = [color] * len(x)
    data_source = bm.ColumnDataSource({ 'x' : x, 'y' : y, 'color': color, **kwargs })

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height)
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show: pl.show(fig)
    return fig

Loading BokehJS ...

In [23]:
draw_vectors(word_vectors_pca[:, 0], word_vectors_pca[:, 1], token=words)

# hover a mouse over there and see if you can identify the clusters

figure(id='p1004', ...)

### Visualizing neighbors with t-SNE
PCA is nice but it's strictly linear and thus only able to capture coarse high-level structure of the data.

If we instead want to focus on keeping neighboring points near, we could use TSNE, which is itself an embedding method. Here you can read __[more on TSNE](https://distill.pub/2016/misread-tsne/)__.

In [28]:
from sklearn.manifold import TSNE

# map word vectors onto 2d plane with TSNE. hint: don't panic it may take a minute or two to fit.
# normalize them as just lke with pca

tsne = TSNE(n_components=2)
word_tsne = tsne.fit_transform(word_vectors)

mean = word_tsne.mean(axis=0)
std = word_tsne.std(axis=0)

word_tsne = (word_tsne - mean) / std

In [29]:
draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color='green', token=words)

figure(id='p1157', ...)

### Visualizing phrases

Word embeddings can also be used to represent short phrases. The simplest way is to take __an average__ of vectors for all tokens in the phrase with some weights.

This trick is useful to identify what data are you working with: find if there are any outliers, clusters or other artefacts.

Let's try this new hammer on our data!


In [30]:
def get_phrase_embedding(phrase):
    """
    Convert phrase to a vector by aggregating it's word embeddings. See description above.
    """
    # 1. lowercase phrase
    # 2. tokenize phrase
    # 3. average word vectors for all words in tokenized phrase
    # skip words that are not in model's vocabulary
    # if all words are missing from vocabulary, return zeros
    
    vector = np.zeros([model.vector_size], dtype='float32')
    
    phrase = phrase.lower()
    
    tokens = tokenizer.tokenize(phrase)
    
    valid_vectors = []
    for token in tokens:
        if token in model.key_to_index:  
            valid_vectors.append(model.get_vector(token))

    if valid_vectors:
        vector = np.mean(valid_vectors, axis=0)  
    else:
        vector = np.zeros([model.vector_size], dtype='float32')  
    
    return vector
        
    

In [31]:
vector = get_phrase_embedding("I'm very sure. This never happened to me before...")

assert np.allclose(vector[::10],
                   np.array([ 0.31807372, -0.02558171,  0.0933293 , -0.1002182 , -1.0278689 ,
                             -0.16621883,  0.05083408,  0.17989802,  1.3701859 ,  0.08655966],
                              dtype=np.float32))
assert np.array_equal(get_phrase_embedding("thisisgibberish"), np.zeros([model.vector_size], dtype='float32')), "corner case for all missing words should be handled as described in the function comments"

In [35]:
# let's only consider ~5k phrases for a first run.
chosen_phrases = data[::len(data) // 1000]

# compute vectors for chosen phrases
phrase_vectors = np.array([get_phrase_embedding(i) for i in chosen_phrases])

In [36]:
assert isinstance(phrase_vectors, np.ndarray) and np.isfinite(phrase_vectors).all()
assert phrase_vectors.shape == (len(chosen_phrases), model.vector_size)

In [37]:
# map vectors into 2d space with pca, tsne or your other method of choice
# don't forget to normalize

phrase_vectors_2d = TSNE().fit_transform(phrase_vectors)

phrase_vectors_2d = (phrase_vectors_2d - phrase_vectors_2d.mean(axis=0)) / phrase_vectors_2d.std(axis=0)

In [38]:
draw_vectors(phrase_vectors_2d[:, 0], phrase_vectors_2d[:, 1],
             phrase=[phrase[:50] for phrase in chosen_phrases],
             radius=20,)

figure(id='p1208', ...)

Finally, let's build a simple "similar question" engine with phrase embeddings we've built.

In [39]:
# compute vector embedding for all lines in data
data_vectors = np.array([get_phrase_embedding(l) for l in data])

In [53]:
def cosine_similarity_manual(vec1, vec2):
    if np.linalg.norm(vec1) == 0 or np.linalg.norm(vec2) == 0:
        return 0  
    
    dot_product = np.dot(vec1, vec2)  
    norm_vec1 = np.linalg.norm(vec1)  
    norm_vec2 = np.linalg.norm(vec2)  
    return dot_product / (norm_vec1 * norm_vec2)

def find_nearest(query, k=10):
    """
    Given a text line (query), return k most similar lines from data, sorted from most to least similar.
    Similarity is measured as cosine similarity between query and line embedding vectors.
    """
    query_vector = get_phrase_embedding(query)  
    
    if len(data_vectors) == 0:
        raise ValueError("data_vectors is empty. Ensure embeddings are precomputed.")
    
    similarities = []
    for vec in data_vectors:
        similarity = cosine_similarity_manual(query_vector, vec)
        similarities.append(similarity)
    
    top_k_indices = np.argsort(similarities)[-k:][::-1]  
    
    top_k_lines = [data[i].strip() for i in top_k_indices]
    
    return top_k_lines

In [54]:
results = find_nearest(query="How do i enter the matrix?", k=10)

print(''.join(results))

assert len(results) == 10 and isinstance(results[0], str)
assert results[0] == 'How do I get to the dark web?\n'
assert results[3] == 'What can I do to save the world?\n'

How do I get to the dark web?What should I do to enter hollywood?How do I use the Greenify app?What can I do to save the world?How do I win this?How do I think out of the box? How do I learn to think out of the box?How do I find the 5th dimension?How do I use the pad in MMA?How do I estimate the competition?What do I do to enter the line of event management?


AssertionError: 

In [55]:
find_nearest(query="How does Trump?", k=10)

['What does Donald Trump think about Israel?',
 'What books does Donald Trump like?',
 'What does Donald Trump think of India?',
 'What does India think of Donald Trump?',
 'What does Donald Trump think of China?',
 'What does Donald Trump think about Pakistan?',
 'What companies does Donald Trump own?',
 'What does Dushka Zapata think about Donald Trump?',
 'How does it feel to date Ivanka Trump?',
 'What does salesforce mean?']

In [56]:
find_nearest(query="Why don't i ask a question myself?", k=10)

["Why don't I get a date?",
 "Why do you always answer a question with a question? I don't, or do I?",
 "Why can't I ask a question anonymously?",
 "Why don't I get a girlfriend?",
 "Why don't I have a boyfriend?",
 "I don't have no question?",
 "Why can't I take a joke?",
 "Why don't I ever get a girl?",
 "Can I ask a girl out that I don't know?",
 "Why don't I have a girlfriend?"]

__Now what?__
* Try running TSNE on all data, not just 1000 phrases
* See what other embeddings are there in the model zoo: `gensim.downloader.info()`
* Take a look at [FastText](https://github.com/facebookresearch/fastText) embeddings
* Optimize `find_nearest` with locality-sensitive hashing: use [nearpy](https://github.com/pixelogik/NearPy) or `sklearn.neighbors`.